In [1]:
!pip install pyfaidx transformers datasets tqdm

import pandas as pd
import requests
from tqdm import tqdm
from pyfaidx import Fasta
from transformers import BertTokenizer, BertForSequenceClassification
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 22.2 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
# ========================
#  📌 第二步：读取 VCF 文件（你需要在 Colab 手动上传）
# ========================
vcf_filename = "ClinVar_Coding_SNV_PB.vcf"  # 你需要替换为自己的 VCF 文件名

''' # 解析 VCF 文件
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")
            chrom, pos, ref, alt = fields[0], int(fields[1]), fields[3], fields[4]
            vcf_data.append([chrom, pos, ref, alt]) '''

# 解析 VCF 文件，同时提取 `INFO` 字段
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")

            # 提取关键字段
            chrom, pos, ref, alt, info = fields[0], int(fields[1]), fields[3], fields[4], fields[7]

            # 将数据存入列表
            vcf_data.append([chrom, pos, ref, alt, info])

# 创建 DataFrame
df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT", "INFO"])


#df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT"])

# 生成 True_Label（1=致病, 0=良性）
#df_vcf["True_Label"] = df_vcf["INFO"]

print("✅ 解析 VCF 完成！")




✅ 解析 VCF 完成！


In [3]:
df_vcf

,CHROM,POS,REF,ALT,INFO
0,11,126275389,C,T,1.0
1,11,126277517,A,G,1.0
2,6,26093215,G,T,1.0
3,2,19945787,T,C,1.0
4,20,25302322,G,A,1.0
...,...,...,...,...,...
136920,5,75416973,G,A,1.0
136921,X,41346238,C,G,1.0
136922,7,140801551,T,C,1.0
136923,9,121314019,A,G,1.0


In [4]:
# ========================
#  📌 第三步：下载 GRCh38 参考基因组
# ========================
!wget -c http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
!gunzip -k hg38.fa.gz

--2025-03-19 16:04:47--  http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
Resolving hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 983659424 (938M) [application/x-gzip]
Saving to: ‘hg38.fa.gz’

hg38.fa.gz          100%[===================>] 938.09M   149MB/s    in 6.7s    

2025-03-19 16:04:54 (140 MB/s) - ‘hg38.fa.gz’ saved [983659424/983659424]



In [13]:
# 加载参考基因组
genome = Fasta("hg38.fa")

# ========================
#  📌 第四步：提取突变上下游 50bp 序列（共 101bp）
# ========================
window = 128

def get_sequence(chrom, pos, ref, alt, flank_size=window):
    """ 获取突变上下游 50bp 的基因组序列（共 101bp） """
    chrom = "chr" + chrom
    start = max(0, pos - flank_size - 1)  # UCSC 坐标是 0-based
    end = pos + flank_size
    seq = genome[chrom][start:end].seq.upper()  # 提取上下游序列
    return seq

# 提取所有突变位点的序列
sequences = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf)):
    chrom, pos, ref, alt = row["CHROM"], row["POS"], row["REF"], row["ALT"]
    seq = get_sequence(chrom, pos, ref, alt)
    sequences.append(seq)

df_vcf["Context_Sequence"] = sequences

def generate_mutant_sequence(row):
    """
    用 ALT 替换 Context_Sequence 中 REF 的位置，生成突变序列
    """
    seq = list(row["Context_Sequence"])  # 转换为列表以进行修改
    mut_pos = window  # 突变发生在中心位置（上下游各 50bp）

    # 进行突变替换（仅支持单碱基突变）
    if seq[mut_pos] == row["REF"]:
        seq[mut_pos] = row["ALT"]
    return "".join(seq)

# 生成突变序列
df_vcf["Mutant_Sequence"] = df_vcf.apply(generate_mutant_sequence, axis=1)

df_vcf.to_csv("processed_vcf_with_sequences.csv", index=False)
print("✅ 突变序列提取完成！")


# ========================
#  📌 第五步：转换为 DNABERT2 的 k-mer 格式
# ========================
from tqdm import tqdm

# 启用 tqdm 进度条
tqdm.pandas()

def generate_kmers(sequence, k=6):
    """ 将序列转换为 k-mer 格式（适用于 DNABERT2） """
    return " ".join([sequence[i:i+k] for i in range(len(sequence) - k + 1)])

# 添加进度条
print("🔄 正在转换 Context_Sequence 为 k-mer 格式...")
df_vcf["Kmer_Sequence"] = df_vcf["Context_Sequence"].progress_apply(lambda x: generate_kmers(x, k=6))

print("🔄 正在转换 Mutant_Sequence 为 k-mer 格式...")
df_vcf["Kmer_Sequence_Mutant"] = df_vcf["Mutant_Sequence"].progress_apply(lambda x: generate_kmers(x, k=6))

# 保存结果
df_vcf.to_csv("dnabert2_input.csv", index=False)
print("✅ k-mer 转换完成！")




100%|██████████| 136925/136925 [00:08<00:00, 15313.44it/s]


✅ 突变序列提取完成！
🔄 正在转换 Context_Sequence 为 k-mer 格式...


100%|██████████| 136925/136925 [00:03<00:00, 35693.50it/s]


🔄 正在转换 Mutant_Sequence 为 k-mer 格式...


100%|██████████| 136925/136925 [00:03<00:00, 35201.15it/s]


✅ k-mer 转换完成！


In [12]:
df = pd.read_csv("processed_vcf_with_sequences.csv")
df = df[['CHROM', 'POS', 'REF', 'ALT', 'INFO', 'Context_Sequence', 'Mutant_Sequence']]
df

,CHROM,POS,REF,ALT,INFO,Context_Sequence,Mutant_Sequence
0,11,126275389,C,T,1.0,GCGAAAGGTCCAGTCCTTGGG,GCGAAAGGTCTAGTCCTTGGG
1,11,126277517,A,G,1.0,CTAGTTGTCAACATGTACTTT,CTAGTTGTCAGCATGTACTTT
2,6,26093215,G,T,1.0,ATAATATTAAGGAAGAGGCAG,ATAATATTAATGAAGAGGCAG
3,2,19945787,T,C,1.0,TGTTTTTACCTCAGGATCCAA,TGTTTTTACCCCAGGATCCAA
4,20,25302322,G,A,1.0,CGGAAGCTTCGAGCTGGTGCG,CGGAAGCTTCAAGCTGGTGCG
...,...,...,...,...,...,...,...
136920,5,75416973,G,A,1.0,TGCTTTAAAAGTTATCGCTTC,TGCTTTAAAAATTATCGCTTC
136921,X,41346238,C,G,1.0,GGCAAGGATTCACTGACCTTA,GGCAAGGATTGACTGACCTTA
136922,7,140801551,T,C,1.0,GTGAAAAACGTTTTTCGTACC,GTGAAAAACGCTTTTCGTACC
136923,9,121314019,A,G,1.0,GCCAAGCTCTACAAGGTGAGC,GCCAAGCTCTGCAAGGTGAGC


In [14]:
df_vcf

,CHROM,POS,REF,ALT,INFO,Context_Sequence,Mutant_Sequence,Kmer_Sequence,Kmer_Sequence_Mutant
0,11,126275389,C,T,1.0,CAGGAAATACAATCCAAGAGCAGAAGTCCTCATCCCTCTTTGTGAG...,CAGGAAATACAATCCAAGAGCAGAAGTCCTCATCCCTCTTTGTGAG...,CAGGAA AGGAAA GGAAAT GAAATA AAATAC AATACA ATAC...,CAGGAA AGGAAA GGAAAT GAAATA AAATAC AATACA ATAC...
1,11,126277517,A,G,1.0,GTGGCTACAGCCTTCCCGAGAACCCCAGTGTTTTGTGCACCCGCAG...,GTGGCTACAGCCTTCCCGAGAACCCCAGTGTTTTGTGCACCCGCAG...,GTGGCT TGGCTA GGCTAC GCTACA CTACAG TACAGC ACAG...,GTGGCT TGGCTA GGCTAC GCTACA CTACAG TACAGC ACAG...
2,6,26093215,G,T,1.0,TCAAAGGCTTTAACTTGCTTTTTCTGTTTTAGAGCCCTCACCGTCT...,TCAAAGGCTTTAACTTGCTTTTTCTGTTTTAGAGCCCTCACCGTCT...,TCAAAG CAAAGG AAAGGC AAGGCT AGGCTT GGCTTT GCTT...,TCAAAG CAAAGG AAAGGC AAGGCT AGGCTT GGCTTT GCTT...
3,2,19945787,T,C,1.0,AGAAATTCATGCAGCCGTGTTGGCACTGCTATTCAACAAACTTCTT...,AGAAATTCATGCAGCCGTGTTGGCACTGCTATTCAACAAACTTCTT...,AGAAAT GAAATT AAATTC AATTCA ATTCAT TTCATG TCAT...,AGAAAT GAAATT AAATTC AATTCA ATTCAT TTCATG TCAT...
4,20,25302322,G,A,1.0,CCTGGGTGGGAAGAGAATGTCTCACCTCAGTATCCGTGGCAGCTCA...,CCTGGGTGGGAAGAGAATGTCTCACCTCAGTATCCGTGGCAGCTCA...,CCTGGG CTGGGT TGGGTG GGGTGG GGTGGG GTGGGA TGGG...,CCTGGG CTGGGT TGGGTG GGGTGG GGTGGG GTGGGA TGGG...
...,...,...,...,...,...,...,...,...,...
136920,5,75416973,G,A,1.0,TATTTTTAAAAGGAAAAAAACCTAGTCTTACCTTATCCAGTCTCTT...,TATTTTTAAAAGGAAAAAAACCTAGTCTTACCTTATCCAGTCTCTT...,TATTTT ATTTTT TTTTTA TTTTAA TTTAAA TTAAAA TAAA...,TATTTT ATTTTT TTTTTA TTTTAA TTTAAA TTAAAA TAAA...
136921,X,41346238,C,G,1.0,AAGCCCGTTTTTAAGAAGATATATATGTATTTTAATTGACACATTA...,AAGCCCGTTTTTAAGAAGATATATATGTATTTTAATTGACACATTA...,AAGCCC AGCCCG GCCCGT CCCGTT CCGTTT CGTTTT GTTT...,AAGCCC AGCCCG GCCCGT CCCGTT CCGTTT CGTTTT GTTT...
136922,7,140801551,T,C,1.0,ATAATTAACACACATCAGTGGAACTTCTGTACTACAACGCTGGTGA...,ATAATTAACACACATCAGTGGAACTTCTGTACTACAACGCTGGTGA...,ATAATT TAATTA AATTAA ATTAAC TTAACA TAACAC AACA...,ATAATT TAATTA AATTAA ATTAAC TTAACA TAACAC AACA...
136923,9,121314019,A,G,1.0,CCCTCACAGCCACCCTTCCTCTCCATCTCTCTATCTCCTACAGGTG...,CCCTCACAGCCACCCTTCCTCTCCATCTCTCTATCTCCTACAGGTG...,CCCTCA CCTCAC CTCACA TCACAG CACAGC ACAGCC CAGC...,CCCTCA CCTCAC CTCACA TCACAG CACAGC ACAGCC CAGC...


In [39]:
# ========================
#  📌 第六步：使用 DNABERT2 进行预测
# =======================


from transformers import BertModel, AutoTokenizer

model = BertModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# ✅ 解决 DNABERT2 的 `config_class` 兼容性问题
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

from tqdm import tqdm  # 导入进度条

def predict_sequence(sequence):
    """ 使用 DNABERT2 计算句子的向量表示 """
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].tolist()[0]  # 取 [CLS] token 的向量表示

# 使用 tqdm 显示进度条
predictions = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting with DNABERT2"):
    pred = predict_sequence(row["Kmer_Sequence"])
    predictions.append(pred)

# 添加预测结果
df_vcf["DNABERT2_Predictions"] = predictions

# 保存结果
df_vcf.to_csv("dnabert2_predictions.csv", index=False)
print("✅ DNABERT2 预测完成！结果已保存：dnabert2_predictions.csv")


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.embeddings.position_embeddings.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.1.attention.self.key.bias', 'bert.encoder.layer.1.attention.self.key.weight', 'bert.encoder.layer.1.attention.self.query

✅ DNABERT2 预测完成！结果已保存：dnabert2_predictions.csv


In [ ]:
predictions_m = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting with DNABERT2 (Mutatnt)"):
    pred_m = predict_sequence(row["Kmer_Sequence_Mutant"])
    predictions_m.append(pred_m)

df_vcf["DNABERT2_Predictions_Mutant"] = predictions_m
# 保存结果
df_vcf.to_csv("dnabert2_predictions.csv", index=False)
print("✅ DNABERT2 预测完成！结果已保存：dnabert2_predictions.csv")

Predicting with DNABERT2 (Mutatnt):  53%|█████▎    | 72415/136925 [2:22:10<1:54:15,  9.41it/s]

In [ ]:
rm -rf ~/.cache/huggingface/hub


In [26]:
df_sample = df_vcf.head(5000)

# 进度条 + 只运行 Sample
predictions = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Predicting with DNABERT2 (Sample)"):
    pred = predict_sequence(row["Kmer_Sequence"])
    predictions.append(pred)

df_sample["DNABERT2_Predictions"] = predictions

predictions_m = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Predicting with DNABERT2 (Sample)"):
    pred_m = predict_sequence(row["Kmer_Sequence_Mutant"])
    predictions_m.append(pred_m)

df_sample["DNABERT2_Predictions_Mutant"] = predictions_m

# 保存 Sample 结果
df_sample.to_csv("dnabert2_sample_predictions.csv", index=False)
print("✅ Sample 预测完成！结果已保存：dnabert2_sample_predictions.csv")


Predicting with DNABERT2 (Sample): 100%|██████████| 5000/5000 [18:44<00:00,  4.45it/s]
<ipython-input-26-5a85c9bcbfbd>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample["DNABERT2_Predictions"] = predictions
Predicting with DNABERT2 (Sample): 100%|██████████| 5000/5000 [18:29<00:00,  4.51it/s]
<ipython-input-26-5a85c9bcbfbd>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample["DNABERT2_Predictions_Mutant"] = predictions_m


✅ Sample 预测完成！结果已保存：dnabert2_sample_predictions.csv


In [32]:
import ast
# 重新导入必要的库
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 重新加载文件
file_path = "dnabert2_sample_predictions.csv"

# 读取 DNABERT2 预测的结果文件
df_predictions = pd.read_csv(file_path)


def compute_llr(prediction_list):
    values = np.array(ast.literal_eval(prediction_list))  # 解析字符串列表
    return np.sum(values)  # 计算总和


# 计算参考序列的 log-likelihood
df_predictions["LLR_Ref"] = df_predictions["DNABERT2_Predictions"].apply(compute_llr)
df_predictions["LLR_Mut"] = df_predictions["DNABERT2_Predictions_Mutant"].apply(compute_llr)

# 计算 LLR 差值
df_predictions["Delta_LLR"] = df_predictions["LLR_Mut"] - df_predictions["LLR_Ref"]

auc_score = roc_auc_score(df_predictions["INFO"], df_predictions["Delta_LLR"])
print(f"Zero-shot 预测 AUC: {auc_score:.4f}")



Zero-shot 预测 AUC: 0.5513


In [28]:
df_predictions

,CHROM,POS,REF,ALT,INFO,Context_Sequence,Mutant_Sequence,Kmer_Sequence,Kmer_Sequence_Mutant,DNABERT2_Predictions,DNABERT2_Predictions_Mutant,LLR_Ref,LLR_Mut,Delta_LLR
0,11,126275389,C,T,1.0,CAGGAAATACAATCCAAGAGCAGAAGTCCTCATCCCTCTTTGTGAG...,CAGGAAATACAATCCAAGAGCAGAAGTCCTCATCCCTCTTTGTGAG...,CAGGAA AGGAAA GGAAAT GAAATA AAATAC AATACA ATAC...,CAGGAA AGGAAA GGAAAT GAAATA AAATAC AATACA ATAC...,"[0.5665522217750549, 0.9618163704872131, 0.090...","[0.5652540922164917, 0.9604885578155518, 0.092...",-1.479719e-06,0.000004,5.714355e-06
1,11,126277517,A,G,1.0,GTGGCTACAGCCTTCCCGAGAACCCCAGTGTTTTGTGCACCCGCAG...,GTGGCTACAGCCTTCCCGAGAACCCCAGTGTTTTGTGCACCCGCAG...,GTGGCT TGGCTA GGCTAC GCTACA CTACAG TACAGC ACAG...,GTGGCT TGGCTA GGCTAC GCTACA CTACAG TACAGC ACAG...,"[0.5685995817184448, 0.976004958152771, 0.1445...","[0.5695587992668152, 0.9789445400238037, 0.147...",6.810296e-06,0.000005,-2.159970e-06
2,6,26093215,G,T,1.0,TCAAAGGCTTTAACTTGCTTTTTCTGTTTTAGAGCCCTCACCGTCT...,TCAAAGGCTTTAACTTGCTTTTTCTGTTTTAGAGCCCTCACCGTCT...,TCAAAG CAAAGG AAAGGC AAGGCT AGGCTT GGCTTT GCTT...,TCAAAG CAAAGG AAAGGC AAGGCT AGGCTT GGCTTT GCTT...,"[0.5280572175979614, 0.9612535834312439, 0.117...","[0.5262215733528137, 0.9570657014846802, 0.112...",-4.002097e-06,0.000016,1.980571e-05
3,2,19945787,T,C,1.0,AGAAATTCATGCAGCCGTGTTGGCACTGCTATTCAACAAACTTCTT...,AGAAATTCATGCAGCCGTGTTGGCACTGCTATTCAACAAACTTCTT...,AGAAAT GAAATT AAATTC AATTCA ATTCAT TTCATG TCAT...,AGAAAT GAAATT AAATTC AATTCA ATTCAT TTCATG TCAT...,"[0.5306355953216553, 0.9675465822219849, 0.105...","[0.5288676619529724, 0.9631557464599609, 0.104...",-1.319556e-05,-0.000009,4.346395e-06
4,20,25302322,G,A,1.0,CCTGGGTGGGAAGAGAATGTCTCACCTCAGTATCCGTGGCAGCTCA...,CCTGGGTGGGAAGAGAATGTCTCACCTCAGTATCCGTGGCAGCTCA...,CCTGGG CTGGGT TGGGTG GGGTGG GGTGGG GTGGGA TGGG...,CCTGGG CTGGGT TGGGTG GGGTGG GGTGGG GTGGGA TGGG...,"[0.5619077086448669, 0.9496036767959595, 0.154...","[0.5589364171028137, 0.9505226016044617, 0.154...",-2.606073e-06,-0.000003,-2.594898e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2,72888457,A,G,1.0,TGTCCAAAGGCTTCGTGGACCTGAGTGACTCCACTCAAGTGAACAA...,TGTCCAAAGGCTTCGTGGACCTGAGTGACTCCACTCAAGTGAACAA...,TGTCCA GTCCAA TCCAAA CCAAAG CAAAGG AAAGGC AAGG...,TGTCCA GTCCAA TCCAAA CCAAAG CAAAGG AAAGGC AAGG...,"[0.5903928279876709, 0.9590005278587341, 0.101...","[0.5914040803909302, 0.9554117918014526, 0.103...",6.314745e-06,-0.000003,-9.027921e-06
4996,2,72888497,C,T,1.0,GAACAACTACTGGGCACTGAACTTGACCTCCATGCTCTGCCTGACT...,GAACAACTACTGGGCACTGAACTTGACCTCCATGCTCTGCCTGACT...,GAACAA AACAAC ACAACT CAACTA AACTAC ACTACT CTAC...,GAACAA AACAAC ACAACT CAACTA AACTAC ACTACT CTAC...,"[0.584827184677124, 0.9471467137336731, 0.1032...","[0.5847710967063904, 0.9473755955696106, 0.106...",1.111522e-05,-0.000005,-1.637510e-05
4997,2,72891502,A,T,1.0,TGGCCCGGGAGACCTCCGTGGACCCAGACATGCGAAAAGGGCTGCA...,TGGCCCGGGAGACCTCCGTGGACCCAGACATGCGAAAAGGGCTGCA...,TGGCCC GGCCCG GCCCGG CCCGGG CCGGGA CGGGAG GGGA...,TGGCCC GGCCCG GCCCGG CCCGGG CCGGGA CGGGAG GGGA...,"[0.5628760457038879, 0.9612202644348145, 0.173...","[0.5601948499679565, 0.9623845815658569, 0.174...",2.799730e-06,-0.000003,-5.300877e-06
4998,19,48703417,G,A,0.0,GCCGGTGCTGCACAGCGCCACGGCCAGCAGGATCCCCTGGCAGAAC...,GCCGGTGCTGCACAGCGCCACGGCCAGCAGGATCCCCTGGCAGAAC...,GCCGGT CCGGTG CGGTGC GGTGCT GTGCTG TGCTGC GCTG...,GCCGGT CCGGTG CGGTGC GGTGCT GTGCTG TGCTGC GCTG...,"[0.5621966123580933, 0.9566695094108582, 0.157...","[0.5625465512275696, 0.9599158763885498, 0.160...",-8.509960e-07,-0.000020,-1.886769e-05


In [12]:
# 查看 DNABERT2_Predictions 的数值范围，检查是否是 log-likelihood 或 logits
import numpy as np

# 取前几行数据，转换为数组
sample_predictions = df_predictions["DNABERT2_Predictions"].iloc[:5].apply(ast.literal_eval)

# 计算均值、最小值、最大值
prediction_stats = {
    "Mean": [np.mean(pred) for pred in sample_predictions],
    "Min": [np.min(pred) for pred in sample_predictions],
    "Max": [np.max(pred) for pred in sample_predictions],
}

# 转换为 DataFrame 方便查看
df_stats = pd.DataFrame(prediction_stats)


In [13]:
df_stats

,Mean,Min,Max
0,-9.555151e-09,-15.597169,2.654371
1,2.320409e-08,-15.594188,2.705306
2,8.917444e-09,-15.561594,2.665403
3,9.907126e-09,-15.651737,2.639766
4,4.625652e-09,-15.569960,2.671898


In [33]:
import numpy as np
import ast
from sklearn.model_selection import train_test_split
# 读取 DNABERT2 生成的 embedding 数据
file_path = "dnabert2_sample_predictions.csv"
df_embeddings = pd.read_csv(file_path)

# 解析 DNABERT2 768 维嵌入
df_embeddings["Embedding_Ref"] = df_embeddings["DNABERT2_Predictions"].apply(ast.literal_eval)
df_embeddings["Embedding_Mut"] = df_embeddings["DNABERT2_Predictions_Mutant"].apply(ast.literal_eval)

# 拼接 Reference 和 Mutant 的 embedding，形成 1536 维向量
X = np.hstack([
    np.vstack(df_embeddings["Embedding_Ref"].values),
    np.vstack(df_embeddings["Embedding_Mut"].values)
])

# 获取真实标签
y = df_embeddings["INFO"].values.astype(int)

# 划分数据集
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"📊 数据集划分完成：训练集 {X_train.shape}, 验证集 {X_val.shape}, 测试集 {X_test.shape}")


📊 数据集划分完成：训练集 (4000, 1536), 验证集 (500, 1536), 测试集 (500, 1536)


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim

# 定义论文中的神经网络结构
class Evo2NN(nn.Module):
    def __init__(self):
        super(Evo2NN, self).__init__()
        self.fc1 = nn.Linear(1536, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(512, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 32)
        self.bn3 = nn.BatchNorm1d(32)

        self.output = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.dropout1(torch.relu(self.bn1(self.fc1(x))))
        x = self.dropout2(torch.relu(self.bn2(self.fc2(x))))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.sigmoid(self.output(x))
        return x

# 初始化模型
model = Evo2NN()
print(model)


Evo2NN(
  (fc1): Linear(in_features=1536, out_features=512, bias=True)
  (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=512, out_features=128, bias=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=32, bias=True)
  (bn3): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [35]:
# 转换数据格式
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# 定义损失函数和优化器
criterion = nn.BCELoss()  # 二分类交叉熵
optimizer = optim.Adam(model.parameters(), lr=3e-4)

# 训练参数
num_epochs = 500
patience = 100  # Early Stopping
best_val_loss = float("inf")
patience_counter = 0

# 训练循环
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    # 预测
    y_pred = model(X_train_tensor)

    # 计算损失
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    # 计算验证损失
    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_val_tensor)
        val_loss = criterion(y_val_pred, y_val_tensor)

    # 打印进度
    if epoch % 1 == 0:
        print(f"Epoch {epoch}: 训练损失={loss.item():.4f}, 验证损失={val_loss.item():.4f}")

    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"⏹️ 早停 (Early Stopping) 在 Epoch {epoch}，最佳验证损失={best_val_loss:.4f}")
            break


Epoch 0: 训练损失=0.7426, 验证损失=0.6857
Epoch 1: 训练损失=0.7287, 验证损失=0.6854
Epoch 2: 训练损失=0.7162, 验证损失=0.6868
Epoch 3: 训练损失=0.7025, 验证损失=0.6859
Epoch 4: 训练损失=0.6946, 验证损失=0.6880
Epoch 5: 训练损失=0.6838, 验证损失=0.6918
Epoch 6: 训练损失=0.6767, 验证损失=0.6952
Epoch 7: 训练损失=0.6704, 验证损失=0.6985
Epoch 8: 训练损失=0.6648, 验证损失=0.7011
Epoch 9: 训练损失=0.6584, 验证损失=0.7026
Epoch 10: 训练损失=0.6530, 验证损失=0.7027
Epoch 11: 训练损失=0.6470, 验证损失=0.7017
Epoch 12: 训练损失=0.6414, 验证损失=0.7003
Epoch 13: 训练损失=0.6362, 验证损失=0.6986
Epoch 14: 训练损失=0.6304, 验证损失=0.6969
Epoch 15: 训练损失=0.6282, 验证损失=0.6949
Epoch 16: 训练损失=0.6232, 验证损失=0.6927
Epoch 17: 训练损失=0.6193, 验证损失=0.6905
Epoch 18: 训练损失=0.6160, 验证损失=0.6880
Epoch 19: 训练损失=0.6120, 验证损失=0.6852
Epoch 20: 训练损失=0.6079, 验证损失=0.6822
Epoch 21: 训练损失=0.6042, 验证损失=0.6794
Epoch 22: 训练损失=0.6002, 验证损失=0.6765
Epoch 23: 训练损失=0.5987, 验证损失=0.6735
Epoch 24: 训练损失=0.5941, 验证损失=0.6709
Epoch 25: 训练损失=0.5907, 验证损失=0.6680
Epoch 26: 训练损失=0.5871, 验证损失=0.6650
Epoch 27: 训练损失=0.5831, 验证损失=0.6619
Epoch 28: 训练损失=0.5804, 验证损失=0.

In [36]:
from sklearn.metrics import roc_auc_score

# 计算测试集 AUC
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor).numpy().flatten()

auc_score = roc_auc_score(y_test, y_test_pred)
print(f"🎯 监督学习 AUC: {auc_score:.4f}")


🎯 监督学习 AUC: 0.4620
